<a href="https://colab.research.google.com/github/sakib-cached/ml_class_work/blob/main/pdf_qa_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF Question Answering Lab Notebook
**Topic:** Retrieval-Augmented Generation (RAG) with PDF Documents

---

## Learning objectives
Build a PDF Question Answering pipeline that:
1. Extracts and cleans text from a PDF
2. Splits the document into semantic chunks
3. Retrieves relevant chunks for a given question (TF-IDF or embeddings)
4. Answers questions using a local model (Track A) or Anthropic API (Track B)
5. Evaluates answers and explores multi-turn conversation

Runtime tip: set runtime to GPU before starting.

---
# Block 1: Setup and Installation

Install required libraries and verify the runtime environment.

In [3]:
# Install all required libraries
!pip install -q pymupdf transformers sentencepiece tiktoken accelerate scikit-learn anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 50.9 MB/s eta 0:00:00


In [4]:
import fitz
import re
import textwrap
import tiktoken
import torch
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

print("Libraries imported successfully")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Libraries imported successfully
GPU available: True
Device: Tesla T4


In [20]:
USE_SAMPLE = False   # Set to False to upload your own PDF

if USE_SAMPLE:
    !wget -q -O sample.pdf https://arxiv.org/pdf/1706.03762  # Attention Is All You Need
    PDF_PATH = "sample.pdf"
    print("Downloaded sample PDF: Attention Is All You Need")
else:
    uploaded = files.upload()
    PDF_PATH = list(uploaded.keys())[0]
    print(f"Uploaded: {PDF_PATH}")

Saving Test PDF (Easy).pdf to Test PDF (Easy).pdf
Uploaded: Test PDF (Easy).pdf


---
# Block 2: PDF Text Extraction & Cleaning

Use PyMuPDF (`fitz`) to extract page-level text, then clean and normalize it.

We preserve page numbers so that retrieved passages can be traced back to their source page.

In [21]:
def extract_text_from_pdf(pdf_path: str) -> dict:
    """
    Extract text from each page of a PDF.
    Returns a dict with metadata and a list of page texts.
    """
    doc = fitz.open(pdf_path)
    result = {
        "num_pages": len(doc),
        "pages": []
    }
    for page_num, page in enumerate(doc):
        text = page.get_text("text")
        result["pages"].append({
            "page": page_num + 1,
            "text": text
        })
    doc.close()
    return result


def clean_text(text: str) -> str:
    """
    Basic cleaning:
    - Collapse multiple newlines and spaces
    - Remove stray page numbers
    - Strip leading/trailing whitespace
    """
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


pdf_data = extract_text_from_pdf(PDF_PATH)
full_text_raw = "\n".join(p['text'] for p in pdf_data['pages'])
full_text = clean_text(full_text_raw)

print(f"Pages       : {pdf_data['num_pages']}")
print(f"Total chars : {len(full_text):,}")
print(f"Total words : {len(full_text.split()):,}")
print("\n--- First 500 chars ---")
print(full_text[:500])

Pages       : 1
Total chars : 2,479
Total words : 390

--- First 500 chars ---
Bangladesh is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, 
and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political 
center but also the economic and cultural heart of the nation. It is one of the most densely 
populated cities in the world and plays a major role in the country’s development. 
 
One of the most important natural features of Bangladesh is the Padma River, which is often 
referred to as the lifeline


In [23]:
enc = tiktoken.get_encoding("cl100k_base")
total_tokens = len(enc.encode(full_text))

print(f"Total tokens: {total_tokens:,}")
print()
print("Model context limits:")

if total_tokens > 0:
    bert_fit = 512 / total_tokens * 100
    gpt_fit = 4096 / total_tokens * 100
    claude_fit = min(200000 / total_tokens * 100, 100)
    print(f"  BERT          : 512 tokens  -> fits {bert_fit:.1f}% of the document")
    print(f"  GPT-3.5       : 4096 tokens -> fits {gpt_fit:.1f}% of the document")
    print(f"  Claude Haiku  : 200K tokens -> fits {claude_fit:.1f}% of the document")
else:
    print("  Cannot calculate fit percentages as no text was extracted from the PDF (total tokens = 0).")
    print(f"  BERT          : 512 tokens  -> fits 0.0% of the document")
    print(f"  GPT-3.5       : 4096 tokens -> fits 0.0% of the document")
    print(f"  Claude Haiku  : 200K tokens -> fits 0.0% of the document")

print()
print("For local QA models, chunking + retrieval is essential.")

Total tokens: 503

Model context limits:
  BERT          : 512 tokens  -> fits 101.8% of the document
  GPT-3.5       : 4096 tokens -> fits 814.3% of the document
  Claude Haiku  : 200K tokens -> fits 100.0% of the document

For local QA models, chunking + retrieval is essential.


---
# Block 3: Chunking

Split the document into overlapping token-aware chunks.
Each chunk carries its page number metadata so answers can cite sources.

```
Document
  -> Chunk 1  (pages 1-2)
  -> Chunk 2  (pages 2-3)   <- 50-token overlap with Chunk 1
  -> Chunk N  (pages N-N+1)
```

In [24]:
def chunk_text_with_pages(pdf_data: dict,
                          max_tokens: int = 400,
                          overlap_tokens: int = 50) -> list[dict]:
    """
    Chunk text while preserving page-number metadata.
    Returns a list of dicts: {text, start_page, end_page, chunk_id}
    """
    enc = tiktoken.get_encoding("cl100k_base")

    # Build a flat token list annotated with page numbers
    all_tokens = []
    all_pages  = []

    for page_info in pdf_data['pages']:
        page_num  = page_info['page']
        page_text = clean_text(page_info['text'])
        tokens    = enc.encode(page_text)
        all_tokens.extend(tokens)
        all_pages.extend([page_num] * len(tokens))

    chunks = []
    start  = 0
    chunk_id = 0

    while start < len(all_tokens):
        end          = min(start + max_tokens, len(all_tokens))
        chunk_tokens = all_tokens[start:end]
        chunk_text   = enc.decode(chunk_tokens)
        start_page   = all_pages[start]
        end_page     = all_pages[end - 1]

        chunks.append({
            "chunk_id"   : chunk_id,
            "text"       : chunk_text,
            "start_page" : start_page,
            "end_page"   : end_page,
            "num_tokens" : len(chunk_tokens)
        })
        start    += max_tokens - overlap_tokens
        chunk_id += 1

    return chunks


chunks = chunk_text_with_pages(pdf_data, max_tokens=400, overlap_tokens=50)

print(f"Total chunks : {len(chunks)}")
print(f"Chunk size   : 400 tokens  |  Overlap: 50 tokens")
print()
for c in chunks[:3]:
    print(f"Chunk {c['chunk_id']:>3} | pages {c['start_page']}-{c['end_page']} | "
          f"{c['num_tokens']} tokens | {c['text'][:70].strip()!r}")

Total chunks : 2
Chunk size   : 400 tokens  |  Overlap: 50 tokens

Chunk   0 | pages 1-1 | 400 tokens | 'Bangladesh is a beautiful country in South Asia, known for its rich cu'
Chunk   1 | pages 1-1 | 153 tokens | '. \n \nThe official currency of Bangladesh is the Bangladeshi Taka. It i'


---
# Block 4: Retrieval — Finding Relevant Chunks

When a question is asked, we score every chunk against the question and return the top-k most relevant ones.

| Method | How it works | Best for |
|---|---|---|
| TF-IDF | Keyword frequency match | Fast, no GPU needed |
| Dense embeddings | Semantic similarity | Better understanding, requires model |

In [25]:
# ── TF-IDF Retriever ──────────────────────────────────────────────────────────

class TFIDFRetriever:
    """
    Lightweight keyword-based retriever using TF-IDF cosine similarity.
    No GPU or model download required.
    """
    def __init__(self, chunks: list[dict]):
        self.chunks = chunks
        self.vectorizer = TfidfVectorizer(
            stop_words='english',
            ngram_range=(1, 2),   # unigrams + bigrams
            max_features=20000
        )
        corpus = [c['text'] for c in chunks]
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus)
        print(f"TF-IDF index built: {len(chunks)} chunks, "
              f"{self.tfidf_matrix.shape[1]:,} features")

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        query_vec = self.vectorizer.transform([query])
        scores    = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_idx   = scores.argsort()[::-1][:top_k]
        results   = []
        for idx in top_idx:
            chunk = self.chunks[idx].copy()
            chunk['score'] = float(scores[idx])
            results.append(chunk)
        return results


retriever = TFIDFRetriever(chunks)

# Quick sanity check
test_q = "What is the attention mechanism?"
top_chunks = retriever.retrieve(test_q, top_k=3)
print(f"\nTop chunks for: '{test_q}'\n")
for c in top_chunks:
    print(f"  Chunk {c['chunk_id']:>3} | pages {c['start_page']}-{c['end_page']} "
          f"| score {c['score']:.3f} | {c['text'][:80].strip()!r}")

TF-IDF index built: 2 chunks, 350 features

Top chunks for: 'What is the attention mechanism?'

  Chunk   1 | pages 1-1 | score 0.000 | '. \n \nThe official currency of Bangladesh is the Bangladeshi Taka. It is used in'
  Chunk   0 | pages 1-1 | score 0.000 | 'Bangladesh is a beautiful country in South Asia, known for its rich cultural her'


---
# Block 5: Choose a Track

| Track | Model | Best for |
|---|---|---|
| A | `deepset/roberta-base-squad2` (local) | Offline / no API key needed |
| B | `claude-haiku-3-5` via Anthropic API | Production-style, conversational |

In [11]:
TRACK = "A"  # Change to "A" to use a local extractive QA model

In [26]:
# ── Track A: Local extractive QA with RoBERTa ─────────────────────────────────
if TRACK == "A":
    from transformers import pipeline

    qa_pipeline = pipeline(
        "question-answering",
        model="deepset/roberta-base-squad2",
        device=0 if torch.cuda.is_available() else -1
    )
    print("Local QA model loaded: deepset/roberta-base-squad2")

    def answer_question_local(question: str, context: str) -> dict:
        """
        Extractive QA: finds the exact answer span inside the context.
        Returns the answer text and a confidence score.
        """
        result = qa_pipeline(
            question=question,
            context=context[:3000],   # RoBERTa max is 512 tokens; truncate safely
            max_answer_len=150
        )
        return result

    # Sanity check
    test_context = top_chunks[0]['text']
    result = answer_question_local(test_q, test_context)
    print(f"\nQ: {test_q}")
    print(f"A: {result['answer']}  (score: {result['score']:.3f})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Local QA model loaded: deepset/roberta-base-squad2

Q: What is the attention mechanism?
A: 
key sectors including agriculture, textiles, and remittances from abroad. 
 
The national animal of Bangladesh is the Royal Bengal Tiger. This majestic animal represents 
strength, courage, and pride. It is mainly found in the Sundarbans and is an important symbol of 
the country’s wildlife heritage. 
   (score: 0.000)


In [ ]:
# ── Track B: Anthropic API (claude-haiku-3-5) ─────────────────────────────────
if TRACK == "B":
    import anthropic
    from google.colab import userdata

    # Store your key in Colab Secrets as  ANTHROPIC_API_KEY
    client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

    def answer_question_api(question: str,
                            context: str,
                            chat_history: list = None) -> str:
        """
        Generative QA via Anthropic API.
        Supports optional multi-turn chat_history for follow-up questions.
        """
        system_prompt = (
            "You are a precise document QA assistant. "
            "Answer the user's question using ONLY the document excerpts provided. "
            "If the answer is not in the excerpts, say so clearly. "
            "Always cite the page numbers of the excerpts you use."
        )

        # Build message list
        messages = list(chat_history or [])

        user_content = (
            f"Document excerpts:\n\n{context}\n\n"
            f"Question: {question}"
        )
        messages.append({"role": "user", "content": user_content})

        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=512,
            system=system_prompt,
            messages=messages
        )
        return response.content[0].text

    # Sanity check
    test_context = "\n\n".join(
        f"[Pages {c['start_page']}-{c['end_page']}]\n{c['text']}"
        for c in top_chunks
    )
    answer = answer_question_api(test_q, test_context)
    print(f"Q: {test_q}")
    print(f"A: {answer}")

---
# Block 6: Full QA Pipeline

Combines retrieval + answering into a single `ask()` function.

```
Question
  -> Retrieve top-k chunks (TF-IDF)
  -> Build context string with page citations
  -> Generate answer (local model or API)
  -> Return answer + source pages
```

In [27]:
def build_context(retrieved_chunks: list[dict]) -> str:
    """Concatenate retrieved chunks into a labelled context string."""
    parts = []
    for c in retrieved_chunks:
        label = f"[Pages {c['start_page']}-{c['end_page']}]"
        parts.append(f"{label}\n{c['text']}")
    return "\n\n".join(parts)


def ask(question: str,
        top_k: int = 5,
        chat_history: list = None,
        verbose: bool = True) -> dict:
    """
    End-to-end QA:
      1. Retrieve top_k relevant chunks
      2. Build context
      3. Generate answer
      4. Return answer dict with metadata
    """
    retrieved = retriever.retrieve(question, top_k=top_k)
    context   = build_context(retrieved)
    source_pages = sorted(set(
        p for c in retrieved
        for p in range(c['start_page'], c['end_page'] + 1)
    ))

    if TRACK == "A":
        # Extractive: concatenate all retrieved texts as one long context
        flat_context = " ".join(c['text'] for c in retrieved)
        result       = answer_question_local(question, flat_context)
        answer_text  = result['answer']
        confidence   = result['score']
    else:
        answer_text = answer_question_api(question, context, chat_history)
        confidence  = retrieved[0]['score']   # retrieval score as proxy

    if verbose:
        print("=" * 60)
        print(f"Q: {question}")
        print("-" * 60)
        print(f"A: {textwrap.fill(answer_text, width=70)}")
        print("-" * 60)
        print(f"Source pages : {source_pages}")
        print(f"Top retrieval score : {confidence:.3f}")
        print("=" * 60)

    return {
        "question"    : question,
        "answer"      : answer_text,
        "source_pages": source_pages,
        "retrieved"   : retrieved
    }


# ── Try it out ────────────────────────────────────────────────────────────────
result = ask("What is the Transformer model?")

Q: What is the Transformer model?
------------------------------------------------------------
A: Bangladeshi Taka
------------------------------------------------------------
Source pages : [1]
Top retrieval score : 0.001


In [ ]:
result = ask("How does multi-head attention work?")

In [ ]:
result = ask("What BLEU score did the Transformer achieve?")

---
# Block 7: Multi-Turn Conversation (Track B only)

Track B uses the Anthropic API which supports stateful chat.
We maintain a `chat_history` list and append each turn to enable follow-up questions.

In [ ]:
if TRACK == "B":
    chat_history = []

    def chat(question: str, top_k: int = 5):
        """Multi-turn QA — remembers previous questions in this session."""
        retrieved   = retriever.retrieve(question, top_k=top_k)
        context     = build_context(retrieved)
        source_pages = sorted(set(
            p for c in retrieved
            for p in range(c['start_page'], c['end_page'] + 1)
        ))

        answer_text = answer_question_api(question, context, chat_history)

        # Update history (store without context to keep tokens manageable)
        chat_history.append({"role": "user",      "content": question})
        chat_history.append({"role": "assistant",  "content": answer_text})

        print(f"\n[Turn {len(chat_history)//2}]")
        print(f"Q: {question}")
        print(f"A: {textwrap.fill(answer_text, width=70)}")
        print(f"   (sources: pages {source_pages})")
        return answer_text

    # Multi-turn example
    chat("What problem does the Transformer solve?")
    chat("Can you elaborate on the positional encoding part?")   # follow-up
    chat("How does this compare to RNNs?")
else:
    print("Multi-turn conversation is only available in Track B.")

---
# Block 8: Interactive Q&A Session

Run this cell for a live command-line Q&A loop.
Type `quit` or `exit` to stop.

In [28]:
print("=" * 60)
print("  PDF Q&A — interactive session")
print(f"  Document: {PDF_PATH}  |  {pdf_data['num_pages']} pages")
print("  Type 'quit' to exit")
print("=" * 60)

session_history = []

while True:
    try:
        q = input("\nYour question: ").strip()
    except EOFError:
        break

    if not q:
        continue
    if q.lower() in ("quit", "exit"):
        print("Session ended.")
        break

    result = ask(q, chat_history=(session_history if TRACK == "B" else None))

    if TRACK == "B":
        session_history.append({"role": "user",     "content": q})
        session_history.append({"role": "assistant", "content": result['answer']})

  PDF Q&A — interactive session
  Document: Test PDF (Easy).pdf  |  1 pages
  Type 'quit' to exit

Your question: what is royal bengal tiger
Q: what is royal bengal tiger
------------------------------------------------------------
A: The national animal of Bangladesh is the Royal Bengal Tiger. This
majestic animal represents  strength, courage, and pride
------------------------------------------------------------
Source pages : [1]
Top retrieval score : 0.222

Your question: what is largest mangrove forest?
Q: what is largest mangrove forest?
------------------------------------------------------------
A: Sundarbans
------------------------------------------------------------
Source pages : [1]
Top retrieval score : 1.413

Your question: which forest is unesco world heritage
Q: which forest is unesco world heritage
------------------------------------------------------------
A: Sundarbans
------------------------------------------------------------
Source pages : [1]
Top retrieval sc

KeyboardInterrupt: Interrupted by user

---
# Block 9: Batch Evaluation

Test the pipeline against a set of known questions and expected answers.
Measures exact-match and keyword overlap as lightweight quality metrics.

| Metric | What it measures |
|---|---|
| Exact match | Is the reference text literally in the answer? |
| Keyword recall | What fraction of key terms appear in the answer? |

In [ ]:
# Define evaluation set — adjust these for your own PDF
eval_questions = [
    {
        "question"   : "What is Scaled Dot-Product Attention?",
        "keywords"   : ["query", "key", "value", "softmax", "dot product"]
    },
    {
        "question"   : "What training data was used?",
        "keywords"   : ["WMT", "English", "German", "French"]
    },
    {
        "question"   : "What optimizer was used?",
        "keywords"   : ["Adam", "learning rate", "warmup"]
    },
    {
        "question"   : "How many encoder layers does the base model have?",
        "keywords"   : ["6", "six", "layers"]
    },
]


def keyword_recall(answer: str, keywords: list[str]) -> float:
    """Fraction of keywords that appear in the answer (case-insensitive)."""
    answer_lower = answer.lower()
    hits = sum(1 for kw in keywords if kw.lower() in answer_lower)
    return hits / len(keywords) if keywords else 0.0


print(f"{'Question':<45} {'KW Recall':>10}")
print("-" * 57)

recall_scores = []
for item in eval_questions:
    result = ask(item['question'], verbose=False)
    recall = keyword_recall(result['answer'], item['keywords'])
    recall_scores.append(recall)
    print(f"{item['question'][:44]:<45} {recall:>9.2f}")

print("-" * 57)
print(f"{'Mean keyword recall':<45} {np.mean(recall_scores):>9.2f}")

---
# Discussion

### What we built
A Retrieval-Augmented Generation (RAG) pipeline:
1. **Extract** — PyMuPDF pulls plain text from each PDF page.
2. **Chunk** — token-aware splitting with overlap preserves context at boundaries.
3. **Index** — TF-IDF vectorises all chunks; cosine similarity finds the most relevant ones at query time.
4. **Generate** — retrieved context is passed to a QA model (extractive or generative).
5. **Evaluate** — keyword recall gives a fast signal on answer quality.

### Limitations and extensions

| Limitation | Possible fix |
|---|---|
| TF-IDF misses synonyms | Replace with dense embeddings (`sentence-transformers`) |
| Extractive QA (Track A) only quotes exact spans | Use Track B for generative answers |
| Fixed chunk size | Experiment with sentence-boundary chunking |
| No reranking | Add a cross-encoder reranker after TF-IDF retrieval |
| No table/figure understanding | Use a multimodal model or dedicated table extractor |